# 충남대 캠퍼스 챗봇 — Colab 데모 (7.8B + CPU 오프로드 + 안 끊기는 프록시)

새 Colab에서 위에서부터 셀을 순서대로 실행하세요. 런타임은 **GPU(T4)** 로 설정.
(메뉴: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 = T4 GPU)

**핵심 설계**
- 생성모델 7.8B(4bit) + bge-m3 임베더/리랭커는 **CPU로 내려 VRAM 확보** → T4에서 OOM 회피
- **cloudflared 안 씀** → Colab 자체 포트 프록시로 노출(100초 한도 없음 = 524 없음)
- 느린 학과서버 크롤 타임아웃 12초 → 라이브 답이 100초 안에

속도 우선이면 4번 셀의 `MODEL_PRIMARY_NAME` 을 `...-2.4B-Instruct` 로 바꾸세요(더 빠르고 더 안전).

## 1. 코드 가져오기 (GitHub clone / pull)

In [ ]:
import os, subprocess
PROJ = '/content/cnu-llm-bot'
REPO = 'https://github.com/Longarden/cnu-llm-bot.git'
if not os.path.isdir(PROJ):
    subprocess.run(['git', 'clone', REPO, PROJ], check=True)
else:
    subprocess.run(['git', '-C', PROJ, 'pull'], check=False)
os.chdir(PROJ)
print('cwd =', os.getcwd())

## 2. 의존성 설치
torch 2.5.1 핀이라 설치 후 **런타임 재시작 안내가 뜨면 한 번 재시작**하고, 이 셀은 건너뛰고 3번부터 다시 실행하세요.

In [ ]:
!pip install -q -r requirements.txt
print('의존성 설치 완료')

## 3. 데이터 복원 (model/ 분류기 가중치 + chroma_db/ 벡터DB)
이 둘은 용량 때문에 깃에 없음 → 구글드라이브에서 가져옴.
본인 드라이브에 `model.tar.gz` 와 `chroma_db.tar.gz`(또는 풀린 `model/`, `chroma_db/` 폴더)를 둔 폴더 경로로 `ASSETS` 를 수정하세요.

In [ ]:
import os, tarfile, shutil
from google.colab import drive
drive.mount('/content/drive')

# 본인 드라이브에서 자산이 있는 폴더로 수정
ASSETS = '/content/drive/MyDrive/cnu_assets'

for name in ('model', 'chroma_db'):
    if os.path.isdir(name) and os.listdir(name):
        print(f'[restore] {name}/ 이미 있음 - 스킵'); continue
    tgz = os.path.join(ASSETS, name + '.tar.gz')
    folder = os.path.join(ASSETS, name)
    if os.path.exists(tgz):
        with tarfile.open(tgz) as t:
            t.extractall('.')
        print(f'[restore] {name}/ <- {tgz}')
    elif os.path.isdir(folder):
        shutil.copytree(folder, name, dirs_exist_ok=True)
        print(f'[restore] {name}/ <- {folder}')
    else:
        print(f'[restore] !!! {name} 자산을 {ASSETS} 에서 못 찾음 - ASSETS 경로 확인')

ready = [d for d in ('model', 'chroma_db') if os.path.isdir(d) and os.listdir(d)]
print('준비된 자산:', ready)

## 4. 환경 설정 + UI 실행 + 안 끊기는 Colab 프록시
모델 로드(워밍업)에 약 1분 걸립니다. '서버 준비됨' 뜬 뒤 아래 iframe에 챗봇이 나옵니다.

In [ ]:
import os, sys, threading, time, socket
sys.path.insert(0, os.getcwd())

# ── 생성 모델: 품질 우선 7.8B(4bit). 속도/안전 우선이면 ...-2.4B-Instruct 로 변경 ──
os.environ['MODEL_PRIMARY_NAME'] = 'LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct'
# ── 임베더/리랭커 CPU로 내려 VRAM 확보(7.8B가 GPU 독차지) ──
os.environ['EMBED_DEVICE']  = 'cpu'
os.environ['RERANK_DEVICE'] = 'cpu'
# ── 로컬 생성 + 라이브크롤 + 리랭커 ON ──
os.environ['GEN_BACKEND']   = 'local'
os.environ['CHAT_REALTIME'] = '1'
os.environ['RERANK']        = '1'
os.environ['GRADIO_SHARE']  = '0'   # cloudflared 미사용(100초 한도 회피)
# ── VRAM 단편화 방지 + 크롤 타임아웃 ──
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CRAWL_TIMEOUT'] = '12'
os.environ['CRAWL_RETRIES'] = '0'

from src.chatbot_ui import launch_app
threading.Thread(target=lambda: launch_app(share=False), daemon=True).start()

PORT = 7860
print('워밍업 대기중... (모델 로드 ~1분)')
for _ in range(420):
    try:
        socket.create_connection(('127.0.0.1', PORT), 0.5).close()
        print('서버 준비됨'); break
    except OSError:
        time.sleep(1)

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height='640')

## 참고
- **GPU 확인:** 아래 셀에서 `!nvidia-smi` — python 프로세스가 **한 개만** 보여야 정상(여러 개면 이전 실행이 안 죽은 것 → 런타임 재시작).
- **안 끊김:** 위 iframe은 Colab 프록시라 cloudflared의 100초 한도가 없음 → 7.8B 느린 답도 524 안 남.
- **느리면:** 4번 셀 `MODEL_PRIMARY_NAME` 을 `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` 로 바꾸면 답이 초 단위(채점은 '말 되면 통과'라 2.4B로 충분).
- **OOM 나면:** 런타임 재시작(이전 프로세스 제거) 후 4번 셀부터 다시. 그래도 빠듯하면 2.4B로.

In [ ]:
!nvidia-smi